# validation check — KMeans (running) -> profitable clusters -> LR, DRAW market

`shots_on_target_foragst` space, draw market. Same method as the goals match check with the
SPACE swapped. The k-sweep at the bottom is an extra check, outside the template.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE (modifiable) — the one place that defines this check's space ------
# The running (point-in-time) season features that describe the match. This is the ONLY
# cell to change to move the whole check to another space.
FEATURES = [
    "hmt_season_shots_on_target_for_avg",
    "hmt_season_shots_on_target_agst_avg",
    "awt_season_shots_on_target_for_avg",
    "awt_season_shots_on_target_agst_avg",
]

In [3]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the draw happened
IMPLIED = "mrkt_draw_impl"      # implied prob = 1 / mrkt odds (profitability + betting)

In [4]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER (modifiable) — KMeans on the running space --------------------
# The segmentation method IS this notebook's identity; a different method = a different notebook.
# Here: fit KMeans on the space features (train only), keep the clusters that beat the market on train,
# label each row by its kept cluster (NaN outside -> not bet). Boundaries and assignment both use the
# running (point-in-time) features. Contract: segmenter(train) -> labeler ; labeler(rows) -> Series.

# Keep the clusters/cells where the target happened MORE often than the market priced it, on the TRAIN
# fold only, so the held-out fold never influences the choice.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def kmeans_profitable_segmenter(features, n_clusters, min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[features].to_numpy())
        kept = keep_profitable(train, pd.Series(km.predict(train[features].to_numpy()), index=train.index), min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[features].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 4   # cluster count (see the k-sweep extra check below)
segmenter = kmeans_profitable_segmenter(FEATURES, K)

# The segmenter's self-description: fit on all matches (in-sample) and show each cluster's edge — size,
# outcome rate vs implied, ROI, centre. In-sample, so a map of WHERE the edge sits, not gated evidence.
# KEEP marks clusters a training fold would keep (positive edge).
_km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(matches[FEATURES].to_numpy())
_lab = _km.predict(matches[FEATURES].to_numpy())
print(f"segment edges (in-sample, k={K})   centre = [{'  '.join(FEATURES)}]")
print(" seg      n   rate%  impl%   edge     ROI   keep   centre")
for s in range(K):
    m = matches[_lab == s]
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    c = _km.cluster_centers_[s]
    print(f" {s:>3} {len(m):>7}  {dr*100:5.1f}  {im*100:5.1f}  {(dr-im)*100:+5.1f}  "
          f"{pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {'KEEP' if dr > im else '    '}  "
          f"[{c[0]:.2f} {c[1]:.2f} {c[2]:.2f} {c[3]:.2f}]")

segment edges (in-sample, k=4)   centre = [hmt_season_shots_on_target_for_avg  hmt_season_shots_on_target_agst_avg  awt_season_shots_on_target_for_avg  awt_season_shots_on_target_agst_avg]
 seg      n   rate%  impl%   edge     ROI   keep   centre
   0    1405   28.1   26.1   +2.0  +0.076  KEEP  [4.18 4.63 3.68 5.43]
   1     988   23.5   25.4   -1.9  -0.076        [4.31 4.47 6.02 3.53]
   2    1341   19.2   22.3   -3.1  -0.139        [5.67 3.52 4.20 4.35]
   3    1799   27.7   29.0   -1.3  -0.044        [3.65 4.85 4.23 4.06]


In [6]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="shots_on_target_foragst_draw_kmeans_match_lr", space="shots_on_target_foragst", target="draw",
              abt_filter="major, gn>8, dropna", segmentation=f"KMeans k={K} (match, running, 4D)",
              segment_filter="train-profitable clusters")

shots_on_target_foragst_draw_kmeans_match_lr   pooled +0.020 (713 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+3/4 seasons)
    2223 +0.163    2324 +0.214    2425 -0.119    2526 +0.045

GATE 2  league — ROI by league / drop best   PASS
    france +0.089    england +0.068    germany +0.043    spain -0.060    italy -0.088
    pooled without best league (france): +0.004

GATE 3  walk-forward — ROI by test season    FAIL
    2425 -0.183    2526 +0.045


## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.

In [8]:
# EXTRA CHECK (below the template) — knob robustness: sweep the cluster count k = 2..8.
# Recomputes the whole honest per-fold pipeline for each k. A real edge survives a range of k; one that
# appears at a single k is fragile. (Not in the template: not every segmenter has a cluster count.)
print(" k    pooled     bets   seasons+")
for k in range(2, 9):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=kmeans_profitable_segmenter(FEATURES, k), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {k:<3} {pooled_roi(b['implied'], b['won']):+.4f}   {len(b):>4}   +{int((ps > 0).sum())}/{len(ps)}")

 k    pooled     bets   seasons+
 2   -0.0134    565   +1/2
 3   +0.0051    662   +2/3
 4   +0.0202    713   +3/4
 5   -0.0764    615   +2/4
 6   +0.0433    701   +3/4
 7   -0.0674    798   +1/4
 8   -0.0167    925   +1/4
